In [ ]:
# Cell 0 — Build GuideScan2 index (run once per genome, then skip)
# If the index already exists this cell will skip automatically

import subprocess, os

GENOME_FASTA = '../data/strainB.fasta'  # ← change to your genome FASTA
INDEX_PREFIX = '../data/strainB_index'  # ← where to save the index

if os.path.exists(f'{INDEX_PREFIX}.gs'):
    print('✅ Index already exists — skip this cell and go to Cell 1.')
else:
    print('Building GuideScan2 index — this may take a few minutes...')
    result = subprocess.run(
        ['guidescan', 'index', '--index', INDEX_PREFIX, GENOME_FASTA],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERROR:', result.stderr)
    else:
        print('✅ Index built. You can now run Cell 1 onwards.')

In [ ]:
# Cell 1 — Imports and configuration

# Ensure all packages are installed in the current kernel
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'matplotlib', 'seaborn', 'biopython', 'rs3', 'forgi'], check=True)

import os
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO

# ── Edit these paths to match your files ──────────────────────────────────────
FASTA     = '../data/strainB.fasta'   # ← change to your genome FASTA
GFF3      = '../data/strainB.gff3'   # ← change to your genome gff3
BED       = '../data/mbnT.bed'       # ← change to your target gene bed
INDEX     = '../data/strainB_index'  # ← change to index name
OUTDIR    = '../results/'
TRACR     = 'both'        # 'Hsu2013', 'Chen2013', or 'both'
TRACR_SEQ = ''            # ← paste your tracrRNA sequence here (RNA or DNA)
MISMATCH  = 3             # GuideScan2 mismatches
THRESHOLD = 1             # GuideScan2 off-target threshold
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTDIR, exist_ok=True)
print('✅ Configuration set.')
print(f'   Genome:  {FASTA}')
print(f'   GFF3:    {GFF3}')
print(f'   BED:     {BED}')
print(f'   Index:   {INDEX}')
print(f'   Output:  {OUTDIR}')

In [ ]:
# Cell 2 — Extract SpCas9 guide sequences from target region

def extract_guides(fasta_path, bed_path):
    with open(bed_path) as f:
        line = f.readline().strip().split('\t')
    chrom  = line[0]
    start  = int(line[1])
    end    = int(line[2])
    gene   = line[3] if len(line) > 3 else 'gene'
    strand = line[5] if len(line) > 5 else '+'

    print(f'Target: {gene} | {chrom}:{start}-{end} ({strand})')

    genome = SeqIO.to_dict(SeqIO.parse(fasta_path, 'fasta'))
    region_seq = genome[chrom].seq[start:end]
    print(f'Region length: {len(region_seq)} bp')

    guides = []

    # Forward strand NGG
    for i in range(len(region_seq) - 22):
        protospacer = region_seq[i:i+20]
        pam         = region_seq[i+20:i+23]
        if str(pam)[1:] == 'GG' and 'N' not in str(protospacer).upper():
            guides.append({
                'guide_sequence' : str(protospacer).upper(),
                'chromosome'     : chrom,
                'start'          : start + i,
                'end'            : start + i + 20,
                'strand'         : '+',
                'pam'            : str(pam).upper(),
                'gene'           : gene
            })

    # Reverse strand NGG
    rev_seq = region_seq.reverse_complement()
    for i in range(len(rev_seq) - 22):
        protospacer = rev_seq[i:i+20]
        pam         = rev_seq[i+20:i+23]
        if str(pam)[1:] == 'GG' and 'N' not in str(protospacer).upper():
            guides.append({
                'guide_sequence' : str(protospacer).upper(),
                'chromosome'     : chrom,
                'start'          : end - (i + 20),
                'end'            : end - i,
                'strand'         : '-',
                'pam'            : str(pam).upper(),
                'gene'           : gene
            })

    df = pd.DataFrame(guides)
    print(f'\nFound {len(df)} candidate guides '
          f'({len(df[df.strand=="+"])} forward, '
          f'{len(df[df.strand=="-"])} reverse)')
    return df

guides_df = extract_guides(FASTA, BED)
guides_df.head()

In [ ]:
# Cell 3 — GuideScan2 off-target scoring

def run_guidescan(guides_df, index_prefix, outdir, mismatches=3, threshold=1):
    # Write kmers file
    kmers_path = os.path.join(outdir, 'kmers.txt')
    with open(kmers_path, 'w') as f:
        f.write('id,sequence,pam,chromosome,position,sense\n')
        for i, row in guides_df.iterrows():
            f.write(f'{i},{row["guide_sequence"]},{row["pam"]},'
                    f'{row["chromosome"]},{row["start"]},{row["strand"]}\n')
    print(f'Written {len(guides_df)} kmers to {kmers_path}')

    # Run GuideScan2
    gs2_output = os.path.join(outdir, 'guidescan2_output.csv')
    cmd = [
        'guidescan', 'enumerate',
        '--mismatches',  str(mismatches),
        '--threshold',   str(threshold),
        '--format',      'csv',
        '--mode',        'succinct',
        '--kmers-file',  kmers_path,
        '--output',      gs2_output,
        index_prefix
    ]
    print(f'Running GuideScan2...')
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print('ERROR:', result.stderr)
        return None

    # Parse and merge
    gs2_df    = pd.read_csv(gs2_output)
    gs2_exact = gs2_df[gs2_df['match_distance'] == 0][['id','specificity']].drop_duplicates('id')
    merged    = guides_df.reset_index(drop=True)
    merged['id'] = merged.index
    merged    = merged.merge(gs2_exact, on='id', how='left')
    merged['specificity'] = merged['specificity'].fillna(0)

    print(f'✅ GuideScan2 complete.')
    print(f'   Specificity range: {merged["specificity"].min():.3f} – {merged["specificity"].max():.3f}')
    return merged

merged_df = run_guidescan(guides_df, INDEX, OUTDIR, MISMATCH, THRESHOLD)
merged_df.head()

In [ ]:
# Cell 4 — RS3 on-target scoring (Hsu2013 and Chen2013)

from rs3.seq import predict_seq

def score_rs3(merged_df, fasta_path, tracr='both'):
    genome = SeqIO.to_dict(SeqIO.parse(fasta_path, 'fasta'))

    sequences_30nt = []
    for _, row in merged_df.iterrows():
        chrom  = row['chromosome']
        start  = int(row['start'])
        end    = int(row['end'])
        strand = row['strand']
        seq    = genome[chrom].seq

        if strand == '+':
            context = seq[start-4 : end+6]
            context_str = str(context).upper()
        else:
            context = seq[start-6 : end+4]
            context_str = str(context.reverse_complement()).upper()

        sequences_30nt.append(context_str if len(context_str) == 30 else 'N' * 30)

    tracr_versions = []
    if tracr in ['Hsu2013', 'both']:
        tracr_versions.append('Hsu2013')
    if tracr in ['Chen2013', 'both']:
        tracr_versions.append('Chen2013')

    for tracr_v in tracr_versions:
        print(f'Scoring with {tracr_v}...')
        scores = np.array(predict_seq(sequences_30nt, sequence_tracr=tracr_v))
        merged_df[f'rs3_{tracr_v}'] = scores
        print(f'  Score range: {scores.min():.3f} – {scores.max():.3f}')

    print('✅ RS3 scoring complete.')
    return merged_df

scored_df = score_rs3(merged_df, FASTA, TRACR)
scored_df[['guide_sequence', 'strand', 'specificity',
           'rs3_Hsu2013', 'rs3_Chen2013']].head()

In [ ]:
# Cell 5 — Secondary structure scoring (custom tracrRNA + guide spacer)

import RNA

def dna_to_rna(seq):
    return seq.upper().replace('T', 'U')

def structure_metrics(sequence_rna):
    md = RNA.md()
    fc = RNA.fold_compound(sequence_rna, md)
    structure, mfe = fc.mfe()
    bp_count = structure.count('(')
    hairpins = 0
    i = 0
    while i < len(structure):
        if structure[i] == '(':
            depth = 0
            for j in range(i, len(structure)):
                if structure[j] == '(':
                    depth += 1
                elif structure[j] == ')':
                    depth -= 1
                    if depth == 0:
                        inner = structure[i+1:j]
                        if '(' not in inner:
                            hairpins += 1
                        break
        i += 1
    return {'mfe': mfe, 'bp_count': bp_count, 'hairpins': hairpins, 'structure': structure}

def seed_base_pairs(full_structure):
    seed_struct = full_structure[:12]
    return seed_struct.count('(') + seed_struct.count(')')

def score_secondary_structure(scored_df, tracr_seq):
    tracr_rna = dna_to_rna(tracr_seq)

    print(f'Folding tracrRNA alone ({len(tracr_rna)} nt)...')
    tracr_metrics = structure_metrics(tracr_rna)
    print(f'  MFE      : {tracr_metrics["mfe"]:.2f} kcal/mol')
    print(f'  bp_count : {tracr_metrics["bp_count"]}')
    print(f'  hairpins : {tracr_metrics["hairpins"]}')
    print(f'  structure: {tracr_metrics["structure"]}')

    mfes, bp_counts, hairpin_counts, seed_pair_counts = [], [], [], []

    print(f'\nScoring spacer + tracrRNA for each guide...')
    for _, row in scored_df.iterrows():
        spacer_rna = dna_to_rna(row['guide_sequence'])
        sgrna_rna  = spacer_rna + tracr_rna
        m = structure_metrics(sgrna_rna)
        seed_bp = seed_base_pairs(m['structure'])
        mfes.append(m['mfe'])
        bp_counts.append(m['bp_count'])
        hairpin_counts.append(m['hairpins'])
        seed_pair_counts.append(seed_bp)

    mfes       = np.array(mfes)
    seed_pairs = np.array(seed_pair_counts)

    raw_penalty   = (-mfes) + 3.0 * seed_pairs
    inv           = raw_penalty.max() - raw_penalty
    denom         = inv.max() - inv.min()
    ss_score_norm = inv / denom if denom > 0 else np.ones(len(inv)) * 0.5

    scored_df['ss_mfe']      = mfes
    scored_df['ss_bp_count'] = bp_counts
    scored_df['ss_hairpins'] = hairpin_counts
    scored_df['ss_seed_bp']  = seed_pairs
    scored_df['ss_score']    = ss_score_norm

    print(f'  MFE range      : {mfes.min():.2f} – {mfes.max():.2f} kcal/mol')
    print(f'  Seed-bp range  : {seed_pairs.min()} – {seed_pairs.max()}')
    print(f'  ss_score range : {ss_score_norm.min():.3f} – {ss_score_norm.max():.3f}')
    print('\n✅ Secondary structure scoring complete.')
    return scored_df, tracr_metrics

scored_df, tracr_ref = score_secondary_structure(scored_df, TRACR_SEQ)
scored_df[['guide_sequence', 'ss_mfe', 'ss_seed_bp', 'ss_score']].head()

In [ ]:
# Cell 6 — Rank guides and save output

def rank_guides(scored_df, tracr='both'):
    sort_cols = ['specificity']
    if 'rs3_Chen2013' in scored_df.columns:
        sort_cols.append('rs3_Chen2013')
    elif 'rs3_Hsu2013' in scored_df.columns:
        sort_cols.append('rs3_Hsu2013')
    if 'ss_score' in scored_df.columns:
        sort_cols.append('ss_score')

    ranked_df = scored_df.sort_values(sort_cols, ascending=False)
    ranked_df = ranked_df.reset_index(drop=True)
    ranked_df.index += 1
    ranked_df.index.name = 'rank'
    return ranked_df

ranked_df = rank_guides(scored_df, TRACR)

# Save outputs
scored_path = os.path.join(OUTDIR, 'guides_scored.csv')
ranked_path = os.path.join(OUTDIR, 'guides_ranked.csv')
scored_df.to_csv(scored_path, index=False)
ranked_df.to_csv(ranked_path)

print(f'✅ Saved scored guides: {scored_path}')
print(f'✅ Saved ranked guides: {ranked_path}')
print(f'\nTop 10 guides:')
display_cols = ['guide_sequence', 'chromosome', 'start', 'end', 'strand',
                'specificity', 'rs3_Hsu2013', 'rs3_Chen2013',
                'ss_score', 'ss_mfe', 'ss_seed_bp']
ranked_df[display_cols].head(10)